In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Fnotebooks%2FUtility_pole_analysis%2Futility_pole_basic_analysis.ipynb?utm_source=cropped_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Image Classification Service with Gemini 3.5 Flash

This notebook demonstrates how to classify images from Imagery Insights using Gemini 3.5 Flash via Vertex AI. It includes concurrent parallel execution using `ThreadPoolExecutor` to process image batches efficiently in minutes.

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'YOUR_PROJECT_ID'  # @param {type:"string"}
REGION = 'global'               # @param {type:"string"}

# BigQuery Configuration
BIGQUERY_DATASET_ID = 'imagery_insights___us'       # @param {type:"string"}
BIGQUERY_TABLE_ID = 'cropped_observations_latest'   # @param {type:"string"}
QUERY_LIMIT = 10                                    # @param {type:"integer"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE"             # @param {type:"string"}

# Model & Concurrency Configuration
MODEL = "gemini-3.5-flash"                          # @param {type:"string"}
THINKING_LEVEL = "HIGH"                             # @param ["MINIMAL", "LOW", "MEDIUM", "HIGH"] {type:"string"}
MAX_CONCURRENCY = 20                                # @param {type:"integer"}

## Imports and Vertex AI Initialization

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import vertexai
from google.cloud import bigquery
from google import genai
from google.genai import types
from google.genai.types import Content, Part

# Resolve GCP Project ID parameter
project_id_arg = PROJECT_ID if (PROJECT_ID and PROJECT_ID != 'YOUR_PROJECT_ID') else None

# Initialize Vertex AI SDK and Gemini Client
vertexai.init(project=project_id_arg, location=REGION)
client = genai.Client(vertexai=True, project=project_id_arg, location=REGION)

## Fetch Image URIs from BigQuery

Next, we'll query a BigQuery table to get the GCS URIs of the images we want to classify.

In [ ]:
# Construct BigQuery SQL query dynamically using parameters
dataset_id = BIGQUERY_DATASET_ID if BIGQUERY_DATASET_ID else 'imagery_insights___us'
table_id = BIGQUERY_TABLE_ID if BIGQUERY_TABLE_ID else 'cropped_observations_latest'
project_id_arg = PROJECT_ID if (PROJECT_ID and PROJECT_ID != 'YOUR_PROJECT_ID') else None

if project_id_arg:
    table_ref = f"`{project_id_arg}.{dataset_id}.{table_id}`"
else:
    table_ref = f"`{dataset_id}.{table_id}`"

BIGQUERY_SQL_QUERY = f"""
SELECT
  *
FROM
  {table_ref}
WHERE asset_type = "{ASSET_TYPE}"
LIMIT {QUERY_LIMIT};
"""

# Execute BigQuery Query
try:
    bigquery_client = bigquery.Client(project=project_id_arg)
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    query_response_data = [dict(row) for row in query_job]
    gcs_uris = [item.get("gcs_uri") for item in query_response_data if item.get("gcs_uri")]

    print(f"Successfully fetched {len(gcs_uris)} GCS URIs:")
    for uri in gcs_uris[:10]:
        print(uri)
    if len(gcs_uris) > 10:
        print(f"... and {len(gcs_uris) - 10} more URIs.")
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")

## Define Image Classification Function

This function takes a GCS URI and a prompt, then uses the Gemini 2.5 Flash model to generate a description of the image.

In [ ]:
def classify_image_with_gemini(gcs_uri: str, prompt: str, model_name: str = MODEL, thinking_lvl: str = THINKING_LEVEL, max_retries: int = 3) -> dict:
    """
    Classifies an image using Gemini 3.5 Flash via Vertex AI SDK by directly passing its GCS URI.
    Includes thinking configuration, token usage cost calculation, and retry logic for rate limits.
    """
    contents = [
        prompt,
        Part(file_data={'file_uri': gcs_uri, 'mime_type': 'image/jpeg'})
    ]
    
    config = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level=thinking_lvl
        )
    )
    
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=model_name, contents=contents, config=config)
            
            prompt_tokens = response.usage_metadata.prompt_token_count if response.usage_metadata else 0
            completion_tokens = response.usage_metadata.candidates_token_count if response.usage_metadata else 0
            total_cost = (prompt_tokens * (0.000075 / 1000)) + (completion_tokens * (0.00030 / 1000))
            
            return {
                "gcs_uri": gcs_uri, 
                "result": response.text, 
                "status": "success",
                "cost": total_cost,
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens
            }
        except Exception as e:
            error_str = str(e)
            if ("429" in error_str or "ResourceExhausted" in error_str or "quota" in error_str.lower()) and attempt < max_retries - 1:
                sleep_time = (2 ** attempt) + 1
                time.sleep(sleep_time)
            else:
                if attempt == max_retries - 1:
                    print(f"Error classifying image from URI {gcs_uri}: {e}")
                return {
                    "gcs_uri": gcs_uri, 
                    "result": f"Classification failed: {e}", 
                    "status": "failed",
                    "cost": 0.0,
                    "prompt_tokens": 0,
                    "completion_tokens": 0
                }
    return {
        "gcs_uri": gcs_uri, 
        "result": "Classification failed after retries.", 
        "status": "failed",
        "cost": 0.0,
        "prompt_tokens": 0,
        "completion_tokens": 0
    }

## Classify Images Concurrently

We pass the GCS URIs to Gemini concurrently using Python's `ThreadPoolExecutor` to achieve high throughput. Parallel execution enables processing large image datasets (e.g. 500+ images) in a few minutes instead of over an hour.

In [ ]:
prompt = """You will be provided with a photo of a utility pole:
{photo_of_utility_pole}

Instructions:

1. Analyze the provided image. If the image does not clearly show a utility pole, return: {"error": "No utility pole detected in the image."}
2. Detect and count the following:
    * Transformers
    * Power lines coming from the pole
    * Street lamps attached to the pole
    * Telephone or junction boxes
3. Assess the overall condition of the pole. Look for visible damage, bird nests, or other issues. If the pole appears to be in good condition, note "OK".
4. Note the material with which the pole is made
5. Determine primary type of pole: Report this in the type field:
  * Street light
  * High tension power transmission
  * electricity pole
  * other
6. Provide your findings in the following JSON format:

```json
{
  "pole_condition": "OK/Damaged/Other Issues",
  "type": <pole_type>,
  "material": <material>,
  "transformers": <number_of_transformers>,
  "power_lines": <number_of_power_lines>,
  "street_lamps": <number_of_street_lamps>,
  "junction_boxes": <number_of_junction_boxes>,
  "additional_notes": "<any_other_observations>"
}
```
"""

results = []
start_time = time.time()

if 'gcs_uris' in locals() and gcs_uris:
    concurrency = MAX_CONCURRENCY if 'MAX_CONCURRENCY' in locals() and MAX_CONCURRENCY else 20
    print(f"Starting classification of {len(gcs_uris)} images using {MODEL} (region: {REGION}) with up to {concurrency} parallel workers...")
    
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        future_to_uri = {
            executor.submit(classify_image_with_gemini, uri, prompt, MODEL, THINKING_LEVEL): uri 
            for uri in gcs_uris
        }
        
        completed_count = 0
        for future in as_completed(future_to_uri):
            completed_count += 1
            res = future.result()
            results.append(res)
            if completed_count % 25 == 0 or completed_count == len(gcs_uris):
                elapsed = time.time() - start_time
                rate = completed_count / elapsed if elapsed > 0 else 0
                print(f"Progress: {completed_count}/{len(gcs_uris)} images processed ({elapsed:.1f}s elapsed, {rate:.1f} imgs/sec)")
                
    total_elapsed = time.time() - start_time
    successful_count = sum(1 for r in results if r["status"] == "success")
    total_cost = sum(r.get("cost", 0.0) for r in results)
    
    print(f"\nFinished classifying {len(results)} images ({successful_count} successful) in {total_elapsed:.2f} seconds ({len(results)/total_elapsed:.1f} imgs/sec).")
    print(f"Total estimated cost: ${total_cost:.5f}")
    
    print("\n--- Sample Classification Results (First 3) ---")
    for res in results[:3]:
        print(f"URI: {res['gcs_uri']}\nResult:\n{res['result']}\nCost: ${res.get('cost', 0.0):.6f}\n" + "-"*50)
else:
    print("No GCS URIs were found to classify.")